# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [84]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [85]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [86]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [87]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [88]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
# general_kb = build_general_kb(n, box_h, box_w, givens)
# r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
# query = atom('Is', r, c, v)

# print(pl_resolution(general_kb, query))    
# print(tt_entails(associate('&', general_kb.clauses), query)) 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


For the full general Sudoku KB, pl_resolution did not return within approximately 30 seconds even for a single cell/value query, so I interrupted it. tt_entails also did not return within approximately 30 seconds and was interrupted. This confirms that, although both methods are sound and complete, neither scales practically to the full Sudoku representation.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [89]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [91]:
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(
        definite_kb,
        atom('Is', r, c, v)
    ) == True

# Soundness
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(
                definite_kb,
                atom('Is', r, c, other_v)
            ) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")

solve_full_grid_fc: 12.79s
solve_full_grid_bc: 4.65s


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**



### (a) General KB

In `build_general_kb`, each possible cell/value assignment is represented by a propositional atom `Is_r_c_v`, meaning that cell `(r,c)` contains value `v`.

Because `PropKB` permits general propositional clauses, the main Sudoku constraints can be represented directly.

For each cell, the requirement that it contains at least one value is encoded as:

$$
Is_{r,c,1} \lor Is_{r,c,2} \lor \cdots \lor Is_{r,c,n}.
$$

The requirement that a cell contains at most one value is encoded pairwise. For any two distinct values $v_1$ and $v_2$:

$$
\neg Is_{r,c,v_1} \lor \neg Is_{r,c,v_2}.
$$

Row, column, and box uniqueness constraints use the same pairwise idea. For example, two different cells in the same row cannot both contain the same value $v$:

$$
\neg Is_{r,c_1,v} \lor \neg Is_{r,c_2,v}.
$$

Equivalent clauses are generated for columns and boxes. Each given value is then added directly to the KB as a fact of the form `Is_r_c_v`.

### (b) Definite / Horn KB

The definite-clause representation requires a different encoding because a definite clause has exactly one positive conclusion. In particular, the direct "at least one value" disjunction used in the general KB cannot be represented in the same form.

The definite KB therefore uses two kinds of positive propositional atoms:

- `Is_r_c_v`: cell `(r,c)` is known to contain value `v`
- `Not_r_c_v`: value `v` has been eliminated from cell `(r,c)`

The first group of rules performs elimination. For example, if a cell is known to contain 5, another cell in the same row cannot contain 5:

$$
Is_{1,1,5} \Rightarrow Not_{1,2,5}.
$$

Similar rules eliminate alternative values from the same cell and eliminate the same value from cells in the same row, column, and box.

The second group consists of last-candidate rules. If all values except one have been eliminated from a cell, the remaining value can be concluded. For example:

$$
Not_{1,1,1} \land Not_{1,1,2} \land \cdots \land Not_{1,1,8}
\Rightarrow Is_{1,1,9}.
$$

The givens are inserted as initial `Is` facts. Forward or backward chaining can then derive additional `Not` and `Is` facts by repeatedly applying these definite rules.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**



In the experiment, `pl_resolution` did not return within approximately 30 seconds for a single cell/value query on the full general Sudoku KB, so the run was interrupted. `tt_entails` likewise did not return within approximately 30 seconds and was interrupted.

These observations do not indicate that either method is logically incorrect. Both methods are sound and complete, but their computational cost makes them impractical for this representation.

For truth-table model checking, if a KB contains $m$ propositional symbols, a direct truth-table procedure may need to consider up to

$$
2^m
$$

possible truth assignments in the worst case.

A standard $9 \times 9$ Sudoku representation already contains hundreds of `Is_r_c_v` atoms, so the model space grows exponentially and becomes extremely large.

Resolution encounters a different scalability problem. It repeatedly resolves compatible pairs of clauses and adds newly generated resolvents. As the clause set grows, the number of possible clause pairs may also grow substantially, and the newly generated clauses can participate in further resolution steps. This can lead to a combinatorial increase in both computation time and memory usage.

Therefore, soundness and completeness alone do not imply practical efficiency. For this Sudoku encoding, the Horn-clause representation together with forward or backward chaining is substantially more practical.

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**



The implemented backward-chaining procedure is goal-driven: it begins with the query and searches backwards for rules whose conclusions could establish that query.

### Pseudocode

```text
BC-ENTAILS(KB, query):

    collect the known facts from KB
    index the rules by their conclusion
    proven = known facts

    repeat:
        record the current number of proven facts
        memo = empty

        if PROVE(query, empty visiting set, memo):
            return True

        if no new facts were proved:
            return False


PROVE(goal, visiting, memo):

    if goal is already proven:
        return True

    if goal has a memoized result:
        return the memoized result

    if goal is already in visiting:
        return False

    add goal to visiting

    for each rule whose conclusion is goal:

        all_premises_proved = True

        for each premise of the rule:
            if PROVE(premise, visiting, memo) is False:
                all_premises_proved = False
                break

        if all_premises_proved:
            remove goal from visiting
            add goal to proven
            memoize goal as True
            return True

    remove goal from visiting
    memoize goal as False
    return False
```

Within a single definite rule, the premises are combined using **AND**, because every premise must be established before the conclusion follows. For example:

$$
A \land B \Rightarrow Q
$$

requires both $A$ and $B$ to be proved.

By contrast, if several different rules have the same conclusion, those rules provide alternative proof paths and therefore behave as **OR** alternatives. For example:

$$
A \land B \Rightarrow Q
$$

and

$$
C \land D \Rightarrow Q
$$

allow $Q$ to be established through either $(A \land B)$ or $(C \land D)$.

To avoid infinite recursion caused by cyclic dependencies, the implementation maintains a `visiting` set containing the goals on the current recursive proof path. If a goal is encountered again while it is already in `visiting`, that proof path is stopped.

A `memo` dictionary is also used to avoid repeatedly solving the same subgoal during one proof attempt. Successfully proved subgoals are retained in a `proven` set. Because the KB contains only finitely many atoms, each retry either adds at least one new proved fact or terminates when no further progress is possible. This provides a finite stopping condition for the implemented procedure.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**



One possible extension is to represent the **Naked Pairs** technique using additional propositional atoms and definite rules.

A Naked Pair occurs when two cells $A$ and $B$ in the same row, column, or box have exactly the same two remaining candidate values $x$ and $y$. In that situation, no other cell $C$ in the same unit can contain either $x$ or $y$.

The current `Is` / `Not` representation records values that have been established or eliminated. However, under ordinary propositional semantics, the absence of `Not_{A,x}` does not by itself prove that $x$ is still a candidate for $A$. Therefore, representing Naked Pairs rigorously would benefit from additional positive atoms that explicitly record candidate or pair information, for example an auxiliary proposition indicating that cells $A$ and $B$ form the candidate pair $\{x,y\}$.

Once such a pair has been established, definite rules could derive eliminations for the other cells in the same unit:

$$
Pair(A,B,x,y) \Rightarrow Not_{C,x}
$$

and

$$
Pair(A,B,x,y) \Rightarrow Not_{C,y}.
$$

Two separate rules are required because a definite clause has only one positive conclusion.

This extension is possible in principle, but it would considerably increase the size of the representation. Rules or auxiliary propositions would be required for many combinations of cell pairs, candidate-value pairs, and Sudoku units. More complex techniques such as X-Wing would lead to an even larger encoding.

Thus, definite clauses can support more advanced Sudoku reasoning when the vocabulary is extended appropriately, but the resulting KB may become much larger and less convenient than the basic elimination and last-candidate representation.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**



### (a)

Backward chaining can have an efficiency advantage when the KB is large but only a relatively small subset of its rules is relevant to a particular query.

Backward chaining is **goal-driven**: it begins with the proposition to be proved and explores only rules that may contribute to that goal. For a query about one particular Sudoku cell/value combination, it may therefore avoid exploring many consequences that are unrelated to that query.

Forward chaining is **data-driven**: it begins with the known facts and propagates their consequences. Depending on the implementation and the query workload, this may derive information that is not directly required for the current query.

In the measured full-grid experiment, forward chaining took approximately **12.40 seconds**, whereas backward chaining took approximately **4.77 seconds**.

For this particular puzzle and implementation, backward chaining was therefore faster. This result should not be interpreted as a general claim that backward chaining is always faster. One contributing factor is that the backward-chaining procedure follows query-relevant proof paths, while the forward-chaining solver repeatedly invokes the provided forward-chaining entailment procedure for candidate queries.

### (b)

Backward chaining does not necessarily retain this advantage when a query depends on a large portion of the KB or when many different queries must be answered.

In such cases, backward chaining may repeatedly explore overlapping proof paths. Forward chaining can instead propagate consequences systematically from the known facts, and the resulting derived knowledge can potentially support many later queries.

The relative performance therefore depends on factors such as the structure of the KB, the number and type of queries, and implementation details. Backward chaining is often attractive for focused queries, while forward chaining may be advantageous when a large fraction of the KB is relevant or when many conclusions are required.

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://...`
